# 01 - Préparation des données

In [ ]:
import pandas as pd
import s3fs

S3        = "s3://fifa-wc-predict-915328198414-eu-west-3-an"
RAW       = f"{S3}/raw"
PROCESSED = f"{S3}/processed"

## 1. Chargement et nettoyage des résultats de matchs

In [5]:
# On charge tous les résultats de matchs internationaux
df = pd.read_csv("../data/results.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Dataset complet : {len(df)} matchs")

Dataset complet : 49287 matchs


In [6]:
# On supprime les matchs sans score, ce sont les matchs de la CDM 2026 pas encore joués
print(f"Matchs sans score : {df['home_score'].isna().sum()}")
df = df.dropna(subset=["home_score", "away_score"]).reset_index(drop=True)
print(f"Après suppression des NaN : {len(df)} matchs")

Matchs sans score : 72
Après suppression des NaN : 49215 matchs


In [7]:
# On garde uniquement les matchs depuis la fin de la CDM 2022
# La finale s'est jouée le 18 décembre 2022, on repart donc du 19
df = df[df["date"] >= "2022-12-19"].reset_index(drop=True)

print(f"Matchs dans le cycle CDM 2026 : {len(df)}")
print(f"Période : {df['date'].min().date()} à {df['date'].max().date()}")
df.tail()

Matchs dans le cycle CDM 2026 : 3465
Période : 2022-12-20 à 2026-03-31


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
3460,2026-03-31,Kosovo,Turkey,0.0,1.0,FIFA World Cup qualification,Pristina,Kosovo,False
3461,2026-03-31,Czech Republic,Denmark,2.0,2.0,FIFA World Cup qualification,Prague,Czech Republic,False
3462,2026-03-31,Cameroon,China PR,2.0,0.0,FIFA Series,Melbourne,Australia,True
3463,2026-03-31,Australia,Curaçao,5.0,1.0,FIFA Series,Melbourne,Australia,False
3464,2026-03-31,Kazakhstan,Comoros,1.0,0.0,FIFA Series,Astana,Kazakhstan,False


## 2. Chargement et traduction du classement FIFA

In [9]:
# On charge le ranking scrapé depuis Transfermarkt
# Les noms de pays sont en français, on va les traduire en anglais
rank = pd.read_csv("../data/fifa_ranking-2022-2026.csv")
rank["rank_date"] = pd.to_datetime(rank["rank_date"])

print(f"Ranking : {len(rank)} lignes sur {rank['rank_date'].nunique()} dates")
print(f"Période : {rank['rank_date'].min().date()} à {rank['rank_date'].max().date()}")
rank.head()

Ranking : 5245 lignes sur 25 dates
Période : 2023-01-09 à 2026-03-18


,rank,country_full,confederation,total_points,rank_date
0,1,Brésil,CONMEBOL,1841,2023-01-09
1,136,Salomon,OFC,1096,2023-01-09
2,137,Rwanda,CAF,1094,2023-01-09
3,138,Éthiopie,CAF,1091,2023-01-09
4,139,Suriname,CONCACAF,1077,2023-01-09


In [10]:
# On traduit les noms de pays français vers anglais
# Transfermarkt utilise les noms en français, results.csv utilise l'anglais
FR_TO_EN = {
    "Afghanistan": "Afghanistan", "Afrique du Sud": "South Africa", "Albanie": "Albania",
    "Algérie": "Algeria", "Allemagne": "Germany", "Andorre": "Andorra", "Angleterre": "England",
    "Angola": "Angola", "Anguilla": "Anguilla", "Antigua-et-Bar.": "Antigua and Barbuda",
    "Arabie saoudite": "Saudi Arabia", "Argentine": "Argentina", "Arménie": "Armenia",
    "Aruba": "Aruba", "Australie": "Australia", "Autriche": "Austria", "Azerbaïdjan": "Azerbaijan",
    "Bahamas": "Bahamas", "Bahreïn": "Bahrain", "Bangladesh": "Bangladesh", "Barbade": "Barbados",
    "Belgique": "Belgium", "Belize": "Belize", "Bermudes": "Bermuda", "Bhoutan": "Bhutan",
    "Biélorussie": "Belarus", "Bolivie": "Bolivia", "Bosnie": "Bosnia and Herzegovina",
    "Botswana": "Botswana", "Brunei": "Brunei", "Brésil": "Brazil", "Bulgarie": "Bulgaria",
    "Burkina Faso": "Burkina Faso", "Burundi": "Burundi", "Bénin": "Benin", "Cambodge": "Cambodia",
    "Cameroun": "Cameroon", "Canada": "Canada", "Cap-Vert": "Cape Verde",
    "Centrafrique": "Central African Republic", "Chili": "Chile", "Chine": "China",
    "Chypre": "Cyprus", "Colombie": "Colombia", "Comores": "Comoros",
    "Congo": "Republic of the Congo", "Corée du Nord": "North Korea", "Corée du Sud": "South Korea",
    "Costa Rica": "Costa Rica", "Croatie": "Croatia", "Cuba": "Cuba", "Curaçao": "Curaçao",
    "Côte d'Ivoire": "Ivory Coast", "Danemark": "Denmark", "Djibouti": "Djibouti",
    "Dominique": "Dominica", "El Salvador": "El Salvador", "Espagne": "Spain",
    "Estonie": "Estonia", "Eswatini": "Eswatini", "Fidji": "Fiji", "Finlande": "Finland",
    "France": "France", "Gabon": "Gabon", "Gambie": "Gambia", "Ghana": "Ghana",
    "Gibraltar": "Gibraltar", "Grenade": "Grenada", "Grèce": "Greece", "Guam": "Guam",
    "Guatémala": "Guatemala", "Guinée": "Guinea", "Guinée équat.": "Equatorial Guinea",
    "Guinée-Bissau": "Guinea-Bissau", "Guyana": "Guyana", "Géorgie": "Georgia",
    "Haïti": "Haiti", "Honduras": "Honduras", "Hong Kong": "Hong Kong", "Hongrie": "Hungary",
    "Inde": "India", "Indonésie": "Indonesia", "Irak": "Iraq", "Iran": "Iran",
    "Irlande": "Republic of Ireland", "Irlande du Nord": "Northern Ireland", "Islande": "Iceland",
    "Israël": "Israel", "Italie": "Italy", "Jamaïque": "Jamaica", "Japon": "Japan",
    "Jordanie": "Jordan", "Kazakhstan": "Kazakhstan", "Kenya": "Kenya",
    "Kirghizistan": "Kyrgyzstan", "Kosovo": "Kosovo", "Koweït": "Kuwait", "Laos": "Laos",
    "Lesotho": "Lesotho", "Lettonie": "Latvia", "Liban": "Lebanon", "Libye": "Libya",
    "Libéria": "Liberia", "Liechtenstein": "Liechtenstein", "Lituanie": "Lithuania",
    "Luxembourg": "Luxembourg", "Macao": "Macau", "Macédoine Nord": "North Macedonia",
    "Madagascar": "Madagascar", "Malaisie": "Malaysia", "Malawi": "Malawi",
    "Maldives": "Maldives", "Mali": "Mali", "Malte": "Malta", "Maroc": "Morocco",
    "Maurice": "Mauritius", "Mauritanie": "Mauritania", "Mexique": "Mexico",
    "Moldavie": "Moldova", "Mongolie": "Mongolia", "Montserrat": "Montserrat",
    "Monténégro": "Montenegro", "Mozambique": "Mozambique", "Myanmar": "Myanmar",
    "N. Calédonie": "New Caledonia", "N. Zélande": "New Zealand", "Namibie": "Namibia",
    "Nicaragua": "Nicaragua", "Niger": "Niger", "Nigéria": "Nigeria", "Norvège": "Norway",
    "Népal": "Nepal", "Oman": "Oman", "Ouganda": "Uganda", "Ouzbékistan": "Uzbekistan",
    "Pakistan": "Pakistan", "Palestine": "Palestine", "Panama": "Panama",
    "Papoua.N.Guinée": "Papua New Guinea", "Paraguay": "Paraguay", "Pays de Galles": "Wales",
    "Pays-Bas": "Netherlands", "Philippines": "Philippines", "Pologne": "Poland",
    "Porto Rico": "Puerto Rico", "Portugal": "Portugal", "Pérou": "Peru", "Qatar": "Qatar",
    "R. dominicaine": "Dominican Republic", "RD Congo": "DR Congo", "Roumanie": "Romania",
    "Russie": "Russia", "Rwanda": "Rwanda", "Saint-Marin": "San Marino",
    "Sainte-Lucie": "Saint Lucia", "Salomon": "Solomon Islands", "Samoa": "Samoa",
    "Samoa a.": "American Samoa", "Serbie": "Serbia", "Seychelles": "Seychelles",
    "Sierra Leone": "Sierra Leone", "Singapour": "Singapore", "Slovaquie": "Slovakia",
    "Slovénie": "Slovenia", "Somalie": "Somalia", "Soudan": "Sudan",
    "Soudan du Sud": "South Sudan", "Sri Lanka": "Sri Lanka",
    "St. Kitts/Nevis": "Saint Kitts and Nevis", "St. Vincent": "Saint Vincent and the Grenadines",
    "Suisse": "Switzerland", "Suriname": "Suriname", "Suède": "Sweden", "Syrie": "Syria",
    "São Tomé-et-P.": "Sao Tome and Principe", "Sénégal": "Senegal", "Tadjikistan": "Tajikistan",
    "Tahiti": "Tahiti", "Taipei Chinois": "Chinese Taipei", "Tanzanie": "Tanzania",
    "Tchad": "Chad", "Tchéquie": "Czech Republic", "Thaïlande": "Thailand",
    "Timor-Leste": "Timor-Leste", "Togo": "Togo", "Tonga": "Tonga",
    "Trinité": "Trinidad and Tobago", "Tunisie": "Tunisia", "Turkménistan": "Turkmenistan",
    "Turques-Caïques": "Turks and Caicos Islands", "Turquie": "Turkey", "Ukraine": "Ukraine",
    "Uruguay": "Uruguay", "Vanuatu": "Vanuatu", "Venezuela": "Venezuela",
    "Vierges a.": "US Virgin Islands", "Vierges b.": "British Virgin Islands",
    "Viêt Nam": "Vietnam", "Yémen": "Yemen", "Zambie": "Zambia", "Zimbabwe": "Zimbabwe",
    "ÉAU": "United Arab Emirates", "Écosse": "Scotland", "Égypte": "Egypt",
    "Équateur": "Ecuador", "États-Unis": "United States", "Éthiopie": "Ethiopia",
    "Îles Caïmans": "Cayman Islands", "Îles Cook": "Cook Islands", "Îles Féroé": "Faroe Islands"
}

rank["country_full"] = rank["country_full"].map(FR_TO_EN)

non_traduits = rank["country_full"].isna().sum()
print(f"Noms non traduits : {non_traduits}")

rank = rank.dropna(subset=["country_full"]).reset_index(drop=True)
print(f"Ranking après traduction : {len(rank)} lignes")

Noms non traduits : 4
Ranking après traduction : 5241 lignes


## 3. Merge des résultats avec le classement FIFA

On utilise `merge_asof` pour associer à chaque match le classement FIFA le plus récent disponible avant la date du match.

In [11]:
# On trie les deux datasets par date, c'est requis par merge_asof
df_sorted = df.sort_values("date").copy()
rank_sorted = rank.sort_values("rank_date").copy()

# On prépare la version home du ranking
rank_home = rank_sorted.rename(columns={
    "rank_date": "date", "country_full": "home_team",
    "rank": "rank_home", "total_points": "total_points_home"
})[["date", "home_team", "rank_home", "total_points_home"]]

# On merge pour l'équipe à domicile avec le classement le plus récent avant la date du match
df_ranked = pd.merge_asof(
    df_sorted, rank_home,
    on="date", by="home_team", direction="backward"
)

# On prépare la version away du ranking
rank_away = rank_sorted.rename(columns={
    "rank_date": "date", "country_full": "away_team",
    "rank": "rank_away", "total_points": "total_points_away"
})[["date", "away_team", "rank_away", "total_points_away"]]

# On merge pour l'équipe visiteuse
df_ranked = pd.merge_asof(
    df_ranked.sort_values("date"), rank_away,
    on="date", by="away_team", direction="backward"
)

# On supprime les matchs pour lesquels on n'a pas de ranking
df_ranked = df_ranked.dropna(subset=["rank_home", "rank_away"]).reset_index(drop=True)

print(f"Dataset après merge : {len(df_ranked)} matchs")
print(f"Matchs perdus au merge : {len(df) - len(df_ranked)} (équipes sans ranking)")
df_ranked.tail()

Dataset après merge : 3152 matchs
Matchs perdus au merge : 313 (équipes sans ranking)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,rank_home,total_points_home,rank_away,total_points_away
3147,2026-03-31,Czech Republic,Denmark,2.0,2.0,FIFA World Cup qualification,Prague,Czech Republic,False,43.0,1487.0,21.0,1617.0
3148,2026-03-31,Peru,Honduras,2.0,2.0,Friendly,Madrid,Spain,True,53.0,1460.0,65.0,1380.0
3149,2026-03-31,United States,Portugal,0.0,2.0,Friendly,Atlanta,United States,False,15.0,1682.0,6.0,1760.0
3150,2026-03-31,Australia,Curaçao,5.0,1.0,FIFA Series,Melbourne,Australia,False,27.0,1574.0,81.0,1303.0
3151,2026-03-31,Kazakhstan,Comoros,1.0,0.0,FIFA Series,Astana,Kazakhstan,False,114.0,1173.0,106.0,1193.0


## 4. Vérifications et sauvegarde

In [12]:
# On vérifie que toutes les équipes de la CDM 2026 ont bien des données dans le dataset
equipes_wc2026 = [
    "Algeria", "Argentina", "Australia", "Austria", "Belgium", "Bosnia and Herzegovina",
    "Brazil", "Canada", "Cape Verde", "Colombia", "Croatia", "Curaçao", "Czech Republic",
    "DR Congo", "Ecuador", "Egypt", "England", "France", "Germany", "Ghana", "Haiti",
    "Iran", "Iraq", "Ivory Coast", "Japan", "Jordan", "Mexico", "Morocco", "Netherlands",
    "New Zealand", "Norway", "Panama", "Paraguay", "Portugal", "Qatar", "Saudi Arabia",
    "Scotland", "Senegal", "South Africa", "South Korea", "Spain", "Sweden", "Switzerland",
    "Tunisia", "Turkey", "United States", "Uruguay", "Uzbekistan"
]

equipes_dans_dataset = set(df_ranked["home_team"].unique()) | set(df_ranked["away_team"].unique())
manquantes = [e for e in equipes_wc2026 if e not in equipes_dans_dataset]

if manquantes:
    print(f"Equipes WC 2026 absentes du dataset : {manquantes}")
else:
    print("Toutes les 48 équipes de la CDM 2026 ont des données dans le dataset")

Toutes les 48 équipes de la CDM 2026 ont des données dans le dataset


In [13]:
# On vérifie qu'il n'y a pas de valeurs manquantes dans les colonnes clés
cols_importantes = ["date", "home_team", "away_team", "home_score", "away_score",
                    "rank_home", "rank_away", "total_points_home", "total_points_away"]
print(df_ranked[cols_importantes].isna().sum())

date                 0
home_team            0
away_team            0
home_score           0
away_score           0
rank_home            0
rank_away            0
total_points_home    0
total_points_away    0
dtype: int64


In [14]:
# On sauvegarde le dataset en CSV pour le notebook 02
os.makedirs("../output", exist_ok=True)
df_ranked.to_csv("../output/dataset_prepared.csv", index=False)

print(f"Dataset sauvegardé : ../output/dataset_prepared.csv")
print(f"Taille finale : {len(df_ranked)} matchs, {len(df_ranked.columns)} colonnes")
print(f"Colonnes : {list(df_ranked.columns)}")

Dataset sauvegardé : ../output/dataset_prepared.csv
Taille finale : 3152 matchs, 13 colonnes
Colonnes : ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'rank_home', 'total_points_home', 'rank_away', 'total_points_away']
